# Questions Using Dataframe APIs

In [0]:
# Spark Configuration
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("PySparkETL") \
    .master("local[*]") \
    .config("spark.executor.memory", "4g") \
    .config("spark.executor.cores", "2") \
    .config("spark.driver.memory", "2g") \
    .config("spark.driver.cores", "1") \
    .config("spark.sql.shuffle.partitions", "200") \
    .getOrCreate()

## Join Questions

In [0]:
bookings_df = spark.read.csv(path="dbfs:/FileStore/bookings.csv",
                    sep=',',
                    header=True, inferSchema=True)
members_df = spark.read.csv(path="dbfs:/FileStore/members.csv",
                    sep=',',
                    header=True, inferSchema=True)
facilities_df = spark.read.csv(path="dbfs:/FileStore/facilities.csv",
                    sep=',',
                    header=True, inferSchema=True)

In [0]:
# How can you produce a list of the start times for bookings by members named 'David Farrell'?

joined_df = bookings_df.join(members_df, bookings_df.memid == members_df.memid)

filtered_df = joined_df.filter((members_df.surname == 'Farrell') & (members_df.firstname == 'David'))

result_df = filtered_df.select(bookings_df.starttime)
result_df.show()


+-------------------+
|          starttime|
+-------------------+
|2012-09-18 09:00:00|
|2012-09-18 17:30:00|
|2012-09-18 13:30:00|
|2012-09-18 20:00:00|
|2012-09-19 09:30:00|
|2012-09-19 15:00:00|
|2012-09-19 12:00:00|
|2012-09-20 15:30:00|
|2012-09-20 11:30:00|
|2012-09-20 14:00:00|
|2012-09-21 10:30:00|
|2012-09-21 14:00:00|
|2012-09-22 08:30:00|
|2012-09-22 17:00:00|
|2012-09-23 08:30:00|
|2012-09-23 17:30:00|
|2012-09-23 19:00:00|
|2012-09-24 08:00:00|
|2012-09-24 16:30:00|
|2012-09-24 12:30:00|
+-------------------+
only showing top 20 rows



In [0]:
# Produce a list of the start times for bookings for tennis courts, for the date '2012-09-21'

joined_df = bookings_df.join(facilities_df, bookings_df.facid == facilities_df.facid)

filtered_df = joined_df.filter(bookings_df.starttime.contains('2012-09-21') & facilities_df.name.contains('Tennis Court')).orderBy('starttime', ascending=True)

result_df = filtered_df.select(bookings_df.starttime, facilities_df.name)
result_df.show()

+-------------------+--------------+
|          starttime|          name|
+-------------------+--------------+
|2012-09-21 08:00:00|Tennis Court 1|
|2012-09-21 08:00:00|Tennis Court 2|
|2012-09-21 09:30:00|Tennis Court 1|
|2012-09-21 10:00:00|Tennis Court 2|
|2012-09-21 11:30:00|Tennis Court 2|
|2012-09-21 12:00:00|Tennis Court 1|
|2012-09-21 13:30:00|Tennis Court 1|
|2012-09-21 14:00:00|Tennis Court 2|
|2012-09-21 15:30:00|Tennis Court 1|
|2012-09-21 16:00:00|Tennis Court 2|
|2012-09-21 17:00:00|Tennis Court 1|
|2012-09-21 18:00:00|Tennis Court 2|
+-------------------+--------------+



In [0]:
# List of all members, including the individual who recommended them (if any). Ensure that results are ordered by (surname, firstname).

members_df.createOrReplaceTempView("members")

# Write and execute the SQL query
result_df = spark.sql("""
    SELECT 
        mem.memid AS member_id,
        mem.surname AS member_surname,
        mem.firstname AS member_firstname,
        rec.surname AS recommender_surname,
        rec.firstname AS recommender_firstname
    FROM 
        members mem
    LEFT JOIN 
        members rec
    ON 
        mem.recommendedby = rec.memid
    ORDER BY 
        member_surname, member_firstname
""")

result_df.show()

+---------+--------------+----------------+-------------------+---------------------+
|member_id|member_surname|member_firstname|recommender_surname|recommender_firstname|
+---------+--------------+----------------+-------------------+---------------------+
|       15|         Bader|        Florence|           Stibbons|               Ponder|
|       12|         Baker|            Anne|           Stibbons|               Ponder|
|       16|         Baker|         Timothy|            Farrell|               Jemima|
|        8|        Boothe|             Tim|             Rownam|                  Tim|
|        5|       Butters|          Gerald|              Smith|               Darren|
|       22|        Coplin|            Joan|              Baker|              Timothy|
|       36|       Crumpet|           Erica|              Smith|                Tracy|
|        7|          Dare|           Nancy|           Joplette|               Janice|
|       28|       Farrell|           David|           

In [0]:
# Produce a list of all members who have used a tennis court

joined1_df = members_df.join(bookings_df, members_df.memid == bookings_df.memid, 'left_outer')

joined2_df = joined1_df.join(facilities_df, facilities_df.facid == joined1_df.facid, 'left_outer').filter(facilities_df.name.contains("Tennis Court"))

results_df = joined2_df.select(members_df.firstname, members_df.surname, facilities_df.name).distinct().orderBy(members_df.firstname)

results_df.show()

+---------+-------+--------------+
|firstname|surname|          name|
+---------+-------+--------------+
|     Anne|  Baker|Tennis Court 1|
|     Anne|  Baker|Tennis Court 2|
|   Burton|  Tracy|Tennis Court 2|
|   Burton|  Tracy|Tennis Court 1|
|  Charles|   Owen|Tennis Court 2|
|  Charles|   Owen|Tennis Court 1|
|   Darren|  Smith|Tennis Court 2|
|    David| Pinker|Tennis Court 1|
|    David|  Jones|Tennis Court 1|
|    David|Farrell|Tennis Court 2|
|    David|Farrell|Tennis Court 1|
|    David|  Jones|Tennis Court 2|
|  Douglas|  Jones|Tennis Court 1|
|    Erica|Crumpet|Tennis Court 1|
| Florence|  Bader|Tennis Court 1|
| Florence|  Bader|Tennis Court 2|
|    GUEST|  GUEST|Tennis Court 1|
|    GUEST|  GUEST|Tennis Court 2|
|   Gerald|Butters|Tennis Court 1|
|   Gerald|Butters|Tennis Court 2|
+---------+-------+--------------+
only showing top 20 rows



In [0]:
# Output a list of all members, including the individual who recommended them (if any), without using any joins

members_with_recommender = members_df.select(
    members_df["memid"],
    F.concat(members_df["firstname"], F.lit(" "), members_df["surname"]).alias("fullname"),
    members_df["recommendedby"].alias("recommender_id")
)

members_with_recommender.createOrReplaceTempView("members")

# Use Spark SQL to find members and their recommender's fullname
results_df = spark.sql("""
    SELECT DISTINCT
        mem.fullname AS member_name,
        rec.fullname AS recommender_name
    FROM 
        members mem
    LEFT JOIN 
        members rec
    ON 
        mem.recommender_id = rec.memid
    ORDER BY 
        member_name
""")

results_df.show()


+--------------------+----------------+
|         member_name|recommender_name|
+--------------------+----------------+
|      Anna Mackenzie|    Darren Smith|
|          Anne Baker| Ponder Stibbons|
|        Burton Tracy|            null|
|        Charles Owen|    Darren Smith|
|        Darren Smith|            null|
|       David Farrell|            null|
|         David Jones| Janice Joplette|
|        David Pinker|  Jemima Farrell|
|       Douglas Jones|     David Jones|
|       Erica Crumpet|     Tracy Smith|
|      Florence Bader| Ponder Stibbons|
|         GUEST GUEST|            null|
|      Gerald Butters|    Darren Smith|
|    Henrietta Rumney| Matthew Genting|
|Henry Worthington...|     Tracy Smith|
| Hyacinth Tupperware|            null|
|          Jack Smith|    Darren Smith|
|     Janice Joplette|    Darren Smith|
|      Jemima Farrell|            null|
|         Joan Coplin|   Timothy Baker|
+--------------------+----------------+
only showing top 20 rows



## Aggregation Questions

In [0]:
# Produce a count of the number of recommendations each member has made. Order by member ID.

recommendations_count = members_df.groupBy("recommendedby").count()
results_df = recommendations_count.orderBy("recommendedby")

results_df.show(truncate=False)

+-------------+-----+
|recommendedby|count|
+-------------+-----+
|null         |9    |
|1            |5    |
|2            |3    |
|3            |1    |
|4            |2    |
|5            |1    |
|6            |1    |
|9            |2    |
|11           |1    |
|13           |2    |
|15           |1    |
|16           |1    |
|20           |1    |
|30           |1    |
+-------------+-----+



In [0]:
# Produce a list of the total number of slots booked per facility

slots_per_facility_df = bookings_df.groupBy("facid").agg(F.sum("slots").alias("slots"))

results_df = slots_per_facility_df.orderBy("facid")

results_df.show()

+-----+-----+
|facid|slots|
+-----+-----+
|    0| 1320|
|    1| 1278|
|    2| 1209|
|    3|  830|
|    4| 1404|
|    5|  228|
|    6| 1104|
|    7|  908|
|    8|  911|
+-----+-----+



In [0]:
# Produce a list of the total number of slots booked per facility in the month of September 2012.

filtered_bookings = bookings_df.filter(bookings_df.starttime.contains('2012-09'))

slots_per_facility = filtered_bookings.groupBy("facid").agg(F.sum("slots").alias("slots"))

results_df = slots_per_facility.orderBy("slots", ascending=True)
results_df.show()

+-----+-----+
|facid|slots|
+-----+-----+
|    5|  122|
|    3|  422|
|    7|  426|
|    8|  471|
|    6|  540|
|    2|  570|
|    1|  588|
|    0|  591|
|    4|  648|
+-----+-----+



In [0]:
# Produce a list of the total number of slots booked per facility per month in the year of 2012

filtered_df = bookings_df.filter(bookings_df.starttime.contains('2012'))

filtered_df = filtered_df.withColumn("month", F.month(F.col("starttime")))

month_slots = filtered_df.groupBy("facid", "month").agg(F.sum("slots")).alias("slots")

results_df = month_slots.orderBy("facid", "month").show()

+-----+-----+----------+
|facid|month|sum(slots)|
+-----+-----+----------+
|    0|    7|       270|
|    0|    8|       459|
|    0|    9|       591|
|    1|    7|       207|
|    1|    8|       483|
|    1|    9|       588|
|    2|    7|       180|
|    2|    8|       459|
|    2|    9|       570|
|    3|    7|       104|
|    3|    8|       304|
|    3|    9|       422|
|    4|    7|       264|
|    4|    8|       492|
|    4|    9|       648|
|    5|    7|        24|
|    5|    8|        82|
|    5|    9|       122|
|    6|    7|       164|
|    6|    8|       400|
+-----+-----+----------+
only showing top 20 rows



In [0]:
# Find the total number of members (including guests) who have made at least one booking.

bookings_df.agg(F.countDistinct(bookings_df.memid)).show()

+------------+
|count(memid)|
+------------+
|          30|
+------------+



In [0]:
# Produce a list of each member name, id, and their first booking after September 1st 2012

joined_df = members_df.join(bookings_df, members_df.memid == bookings_df.memid, 'left_outer')
filtered_df = joined_df.filter(bookings_df.starttime > '2012-09-01')

result_df = filtered_df.groupBy(members_df.firstname, members_df.surname, members_df.memid).agg(F.min(bookings_df.starttime).alias("first_booking"))

result_df = result_df.select(members_df.firstname, members_df.surname, members_df.memid, 'first_booking').orderBy(members_df.memid).show()

+---------+---------+-----+-------------------+
|firstname|  surname|memid|      first_booking|
+---------+---------+-----+-------------------+
|    GUEST|    GUEST|    0|2012-09-01 08:00:00|
|   Darren|    Smith|    1|2012-09-01 09:00:00|
|    Tracy|    Smith|    2|2012-09-01 11:30:00|
|      Tim|   Rownam|    3|2012-09-01 16:00:00|
|   Janice| Joplette|    4|2012-09-01 15:00:00|
|   Gerald|  Butters|    5|2012-09-02 12:30:00|
|   Burton|    Tracy|    6|2012-09-01 15:00:00|
|    Nancy|     Dare|    7|2012-09-01 12:30:00|
|      Tim|   Boothe|    8|2012-09-01 08:30:00|
|   Ponder| Stibbons|    9|2012-09-01 11:00:00|
|  Charles|     Owen|   10|2012-09-01 11:00:00|
|    David|    Jones|   11|2012-09-01 09:30:00|
|     Anne|    Baker|   12|2012-09-01 14:30:00|
|   Jemima|  Farrell|   13|2012-09-01 09:30:00|
|     Jack|    Smith|   14|2012-09-01 11:00:00|
| Florence|    Bader|   15|2012-09-01 10:30:00|
|  Timothy|    Baker|   16|2012-09-01 15:00:00|
|    David|   Pinker|   17|2012-09-01 08

## String and Date Questions

In [0]:
# Output the names of all members, formatted as 'Surname, Firstname'

members_df.select(F.concat(members_df.surname, F.lit(","), members_df.firstname)).show()

+-----------------------------+
|concat(surname, ,, firstname)|
+-----------------------------+
|                  GUEST,GUEST|
|                 Smith,Darren|
|                  Smith,Tracy|
|                   Rownam,Tim|
|              Joplette,Janice|
|               Butters,Gerald|
|                 Tracy,Burton|
|                   Dare,Nancy|
|                   Boothe,Tim|
|              Stibbons,Ponder|
|                 Owen,Charles|
|                  Jones,David|
|                   Baker,Anne|
|               Farrell,Jemima|
|                   Smith,Jack|
|               Bader,Florence|
|                Baker,Timothy|
|                 Pinker,David|
|              Genting,Matthew|
|               Mackenzie,Anna|
+-----------------------------+
only showing top 20 rows



In [0]:
# Perform a case-insensitive search to find all facilities whose name begins with 'tennis'

facilities_df.filter(facilities_df.name.rlike("^Tennis")).show()

+-----+--------------+----------+---------+-------------+------------------+
|facid|          name|membercost|guestcost|initialoutlay|monthlymaintenance|
+-----+--------------+----------+---------+-------------+------------------+
|    0|Tennis Court 1|       5.0|     25.0|        10000|               200|
|    1|Tennis Court 2|       5.0|     25.0|         8000|               200|
+-----+--------------+----------+---------+-------------+------------------+



In [0]:
# Find all the telephone numbers that contain parentheses, returning the member ID and telephone number sorted by member ID.

members_df.select(members_df.memid, members_df.telephone).filter(members_df.telephone.rlike("\(\d*\)\d*")).orderBy(members_df.memid).show()

+-----+--------------+
|memid|     telephone|
+-----+--------------+
|    0|(000) 000-0000|
|    3|(844) 693-0723|
|    4|(833) 942-4710|
|    5|(844) 078-4130|
|    6|(822) 354-9973|
|    7|(833) 776-4001|
|    8|(811) 433-2547|
|    9|(833) 160-3900|
|   10|(855) 542-5251|
|   11|(844) 536-8036|
|   13|(855) 016-0163|
|   14|(822) 163-3254|
|   15|(833) 499-3527|
|   20|(811) 972-1377|
|   21|(822) 661-2898|
|   22|(822) 499-2232|
|   24|(822) 413-1470|
|   27|(822) 989-8876|
|   28|(855) 755-9876|
|   29|(855) 894-3758|
+-----+--------------+
only showing top 20 rows



In [0]:
# Produce a count of how many members you have whose surname starts with each letter of the alphabet.

members_firstletter = members_df.withColumn("first_letter", F.substring(F.col("surname"), 1, 1))

result_df = members_firstletter.groupBy("first_letter").agg(F.count("first_letter")).orderBy("first_letter").show()

+------------+-------------------+
|first_letter|count(first_letter)|
+------------+-------------------+
|           B|                  5|
|           C|                  2|
|           D|                  1|
|           F|                  2|
|           G|                  2|
|           H|                  1|
|           J|                  3|
|           M|                  1|
|           O|                  1|
|           P|                  2|
|           R|                  2|
|           S|                  6|
|           T|                  2|
|           W|                  1|
+------------+-------------------+



In [0]:
# Produce a list of all the dates in October 2012.

result_df = bookings_df.select(bookings_df.starttime)
result_df.filter(bookings_df.starttime.contains("2012-10")).show()

+---------+
|starttime|
+---------+
+---------+



In [0]:
# Return a count of bookings for each month, sorted by month

filtered_df = bookings_df.withColumn("month", F.month(F.col("starttime"))).withColumn("year", F.year(F.col("starttime")))

result_df = filtered_df.groupBy("month", "year").agg(F.count("starttime")).alias("monthly_bookings_count").orderBy("year","month")

result_df.show()

+-----+----+----------------+
|month|year|count(starttime)|
+-----+----+----------------+
|    7|2012|             658|
|    8|2012|            1472|
|    9|2012|            1913|
|    1|2013|               1|
+-----+----+----------------+

